In [8]:
import xarray as xr
from herbie import Herbie
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

def fetch_pa_only(date_str, label):
    print(f"--- Processing {label} for {date_str} ---")
    
    H = Herbie(
        date_str,
        model="ifs",
        product="oper",
        fxx=12,
        overwrite=True
    )
    
    # 1. Download and load the list of datasets
    ds_list = H.xarray() 
    
    if isinstance(ds_list, list):
        ds_combined = xr.merge(ds_list, compat='override')
    else:
        ds_combined = ds_list

    # 2. LAT/LON SLICING FOR PENNSYLVANIA
    # PA Bounds: Lat [39.7, 42.5], Lon [-80.5, -74.7]
    # Check if longitude is 0-360 or -180-180
    if ds_combined.longitude.max() > 180:
        lon_min, lon_max = 360 - 80.5, 360 - 74.7
    else:
        lon_min, lon_max = -80.5, -74.7

    print(f"Slicing spatial grid for Pennsylvania...")
    ds_pa = ds_combined.sel(
        latitude=slice(42.5, 39.7), # North to South for most GRIB files
        longitude=slice(lon_min, lon_max)
    )

    # 3. Save the sliced NetCDF
    out_name = f"ifs_{label}_PA.nc"
    ds_pa.to_netcdf(out_name)
    print(f"✅ Saved PA-only data to: {out_name}")
    print(f"New Grid Shape: {ds_pa.t2m.shape}")

if __name__ == "__main__":
    fetch_pa_only("2026-01-25 00:00", "storm")
    fetch_pa_only("2026-04-04 00:00", "calm")

--- Processing storm for 2026-01-25 00:00 ---
✅ Found ┊ model=ifs ┊ product=oper ┊ 2026-Jan-25 00:00 UTC F12 ┊ GRIB2 @ google ┊ IDX @ google


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [10] xarray.Datasets because cfgrib opened with multiple hypercubes.
Slicing spatial grid for Pennsylvania...
✅ Saved PA-only data to: ifs_storm_PA.nc
New Grid Shape: (12, 24)
--- Processing calm for 2026-04-04 00:00 ---
✅ Found ┊ model=ifs ┊ product=oper ┊ 2026-Apr-04 00:00 UTC F12 ┊ GRIB2 @ google ┊ IDX @ google


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [10] xarray.Datasets because cfgrib opened with multiple hypercubes.
Slicing spatial grid for Pennsylvania...
✅ Saved PA-only data to: ifs_calm_PA.nc
New Grid Shape: (12, 24)


In [ ]:
from timezonefinder import TimezoneFinder
import pytz

tf = TimezoneFinder()

def get_timezone_for_location(latitude, longitude):
    tz_name = tf.timezone_at(lng=longitude, lat=latitude)
    if tz_name:
        return pytz.timezone(tz_name)
    else:
        # Default to UTC if no specific timezone is found, or a reasonable default
        return pytz.utc


In [8]:
def calculate_wbgt_from_ecmwf(T, W, sc_lat, sc_lon):
    # 1. Load and Merge ECMWF Datasets
    ds_main = xr.open_dataset(T)
    ds_wind = xr.open_dataset(W)
    ds = xr.merge([ds_main, ds_wind])

    # 2. Select the location (State College, PA)
    point = ds.sel(latitude=sc_lat, longitude=sc_lon, method="nearest")

    # 3. Extract and Convert Variables
    # Temperature: Kelvin to Fahrenheit
    temp_f = (point['t2m'] - 273.15) * 9/5 + 32
    td_f = (point['d2m'] - 273.15) * 9/5 + 32
    
    # Wind: m/s to mph (using gust or 10m wind)
    wind_mph = point['fg10'] * 2.237
    
    # Pressure: Pa to hPa (mb)
    pressure_mb = point['sp'] / 100 if 'sp' in point else 1013.25
    
    # Solar Radiation Flux (W/m2)
    # ECMWF 'ssrd' is accumulated (J/m2). We calculate the difference between timesteps.
    # We assume a 1-hour (3600s) accumulation window.
    ssrd_diff = point['ssrd'].diff('valid_time') / 3600
    # Fill the first step which is lost during diff
    srad_flux = ssrd_diff.reindex(valid_time=point.valid_time, method='bfill')

    # 4. Calculate Relative Humidity (RH)
    # Necessary for the Wet Bulb component
    e = 6.112 * np.exp((17.67 * (td_f - 32) * 5/9) / ((td_f - 32) * 5/9 + 243.5))
    es = 6.112 * np.exp((17.67 * (temp_f - 32) * 5/9) / ((temp_f - 32) * 5/9 + 243.5))
    rh = (e / es) * 100

    # 5. Solve for Wet Bulb (Tw) using Stull's Formula
    # This is the 'Psychrometric' Wet Bulb
    tw_f = temp_f * np.arctan(0.151977 * (rh + 8.313659)**0.5) + \
           np.arctan(temp_f + rh) - np.arctan(rh - 1.676331) + \
           0.00391838 * rh**1.5 * np.arctan(0.023101 * rh) - 4.686035
           
    # 6. Estimate Globe Temperature (Tg) 
    # Accounts for air temp, solar heating, and wind cooling
    tg_f = temp_f + (srad_flux / 25) - (wind_mph / 1.8)

    # 7. Apply the Final Formula: WBGT = 0.7Tw + 0.2Tg + 0.1Td
    # Note: To match OSHA '86', the 0.1 weight uses Dry Bulb (temp_f)
    wbgt = (0.7 * tw_f) + (0.2 * tg_f) + (0.1 * temp_f)

    # 8. Organize into a readable DataFrame
    results = pd.DataFrame({
        'Valid_Time': point.valid_time.values,
        'Dry_Bulb_F': temp_f.values,
        'Dew_Point_F': td_f.values,
        'Wet_Bulb_F': tw_f.values,
        'Globe_Temp_F': tg_f.values,
        'WBGT': wbgt.values
    })

    return results

# --- RUNNING THE CALCULATION ---
# Coordinates for State College, PA
sc_lat, sc_lon = 40.79, -77.86

# filenames as provided in your directory
file1 = "Marecmwf_20260323Mar.nc"
file2 = "10fgMarecmwf_20260323Mar.nc"

df_wbgt = calculate_wbgt_from_ecmwf(file1, file2, sc_lat, sc_lon)

# Display the first few hours
print(df_wbgt.head())

NameError: name 'xr' is not defined